In [11]:
from pathlib import Path

# Your actual project location
PROJECT_DIR = Path(
    r"C:\Users\adity\Downloads\Historical data\BharatMarket_AI"
)

# Project folders
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

OUTPUT_DIR = PROJECT_DIR / "outputs"
CHART_DIR = OUTPUT_DIR / "charts"
REPORT_DIR = OUTPUT_DIR / "reports"
MODEL_DIR = OUTPUT_DIR / "models"

# Create all directories
for folder in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    CHART_DIR,
    REPORT_DIR,
    MODEL_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project path:", PROJECT_DIR)
print("Raw data path:", RAW_DATA_DIR)
print("All project directories created successfully!")

Project path: C:\Users\adity\Downloads\Historical data\BharatMarket_AI
Raw data path: C:\Users\adity\Downloads\Historical data\BharatMarket_AI\data\raw
All project directories created successfully!


In [12]:
import shutil

EXPECTED_FILES = [
    "NIFTY50.csv",
    "BANKNIFTY.csv",
    "RELIANCE.csv",
    "TCS.csv",
    "HDFCBANK.csv",
    "INFY.csv",
    "ICICIBANK.csv",
    "SBIN.csv",
    "ITC.csv",
    "LT.csv"
]

SEARCH_ROOTS = [
    PROJECT_DIR,
    PROJECT_DIR.parent
]

found_files = {}

for filename in EXPECTED_FILES:

    for search_root in SEARCH_ROOTS:

        matches = list(search_root.rglob(filename))

        matches = [
            p for p in matches
            if p.is_file()
        ]

        if matches:
            found_files[filename] = matches[0]
            break

for filename in EXPECTED_FILES:

    if filename in found_files:

        source = found_files[filename]
        destination = RAW_DATA_DIR / filename

        if source.resolve() != destination.resolve():
            shutil.copy2(source, destination)

        print("FOUND:", filename)

    else:
        print("MISSING:", filename)

print("\nTotal CSV files found:", len(found_files), "/ 10")

FOUND: NIFTY50.csv
FOUND: BANKNIFTY.csv
FOUND: RELIANCE.csv
FOUND: TCS.csv
FOUND: HDFCBANK.csv
FOUND: INFY.csv
FOUND: ICICIBANK.csv
FOUND: SBIN.csv
FOUND: ITC.csv
FOUND: LT.csv

Total CSV files found: 10 / 10


# BharatMarket AI
## Indian Stock Market Analysis and Forecasting System

**Course:** CS3216 - Time Series Analysis and Forecasting

### Project Objectives
1. Analyse NIFTY 50 and BANK NIFTY.
2. Analyse 8 major NSE-listed companies.
3. Study historical stock market trends.
4. Apply statistical forecasting models.
5. Develop machine learning forecasting models.
6. Evaluate forecasting accuracy.
7. Develop a historical trading signal engine.
8. Backtest trading strategies.

### Historical Data
Period: November 2021 – November 2026
Frequency: Daily
Instruments: 10

In [1]:
import os
import warnings
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["axes.titlesize"] = 15

print("BharatMarket AI - Environment Ready")
print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)

Matplotlib is building the font cache; this may take a moment.


BharatMarket AI - Environment Ready
Pandas version: 3.1.1
NumPy version: 2.5.5


In [2]:
PROJECT_DIR = Path(r"D:\BharatMarket_AI")

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"

OUTPUT_DIR = PROJECT_DIR / "outputs"
CHART_DIR = OUTPUT_DIR / "charts"
REPORT_DIR = OUTPUT_DIR / "reports"
MODEL_DIR = OUTPUT_DIR / "models"

# Create folders if they don't exist
for folder in [
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    CHART_DIR,
    REPORT_DIR,
    MODEL_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("All folders created successfully!")

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'D:\\'

In [3]:
INSTRUMENTS = {
    "NIFTY50": "NIFTY 50",
    "BANKNIFTY": "BANK NIFTY",
    "RELIANCE": "Reliance Industries",
    "TCS": "Tata Consultancy Services",
    "HDFCBANK": "HDFC Bank",
    "INFY": "Infosys",
    "ICICIBANK": "ICICI Bank",
    "SBIN": "State Bank of India",
    "ITC": "ITC Limited",
    "LT": "Larsen & Toubro"
}

print("Total instruments:", len(INSTRUMENTS))

for symbol, company in INSTRUMENTS.items():
    print(symbol, ":", company)

Total instruments: 10
NIFTY50 : NIFTY 50
BANKNIFTY : BANK NIFTY
RELIANCE : Reliance Industries
TCS : Tata Consultancy Services
HDFCBANK : HDFC Bank
INFY : Infosys
ICICIBANK : ICICI Bank
SBIN : State Bank of India
ITC : ITC Limited
LT : Larsen & Toubro


In [4]:
market_data = {}

for symbol, company in INSTRUMENTS.items():

    file_path = RAW_DATA_DIR / f"{symbol}.csv"

    if not file_path.exists():
        print(f"Missing file: {symbol}.csv")
        continue

    df = pd.read_csv(file_path)

    df.columns = [
        col.strip().lower()
        for col in df.columns
    ]

    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce"
    )

    df = df.sort_values("date").reset_index(drop=True)

    market_data[symbol] = df

    print(
        f"{symbol}: {len(df)} records loaded successfully"
    )

print("\nTotal datasets loaded:", len(market_data))

Missing file: NIFTY50.csv
Missing file: BANKNIFTY.csv
Missing file: RELIANCE.csv
Missing file: TCS.csv
Missing file: HDFCBANK.csv
Missing file: INFY.csv
Missing file: ICICIBANK.csv
Missing file: SBIN.csv
Missing file: ITC.csv
Missing file: LT.csv

Total datasets loaded: 0


In [5]:
validation_results = []

required_columns = [
    "date",
    "open",
    "high",
    "low",
    "close",
    "adj_close",
    "volume"
]

for symbol, df in market_data.items():

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    result = {
        "Symbol": symbol,
        "Rows": len(df),
        "Start Date": df["date"].min(),
        "End Date": df["date"].max(),
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Dates": int(df["date"].duplicated().sum()),
        "Missing Columns": ", ".join(missing_columns)
    }

    validation_results.append(result)

validation_df = pd.DataFrame(validation_results)

display(validation_df)

""


In [6]:
quality_results = []

for symbol, df in market_data.items():

    invalid_ohlc = (
        (df["high"] < df["low"]) |
        (df["high"] < df["open"]) |
        (df["high"] < df["close"]) |
        (df["low"] > df["open"]) |
        (df["low"] > df["close"])
    )

    invalid_prices = (
        (df[["open", "high", "low", "close"]] <= 0)
        .any(axis=1)
    )

    invalid_volume = df["volume"] < 0

    quality_results.append({
        "Symbol": symbol,
        "Invalid OHLC Rows": int(invalid_ohlc.sum()),
        "Invalid Price Rows": int(invalid_prices.sum()),
        "Negative Volume Rows": int(invalid_volume.sum())
    })

quality_df = pd.DataFrame(quality_results)

display(quality_df)

""


In [7]:
nifty = market_data["NIFTY50"]

display(nifty.head(10))

KeyError: 'NIFTY50'

In [8]:
plt.figure(figsize=(16, 6))

plt.plot(
    nifty["date"],
    nifty["close"],
    label="NIFTY 50 Closing Price",
    linewidth=1.5
)

plt.title("NIFTY 50 - Five Years Historical Price")
plt.xlabel("Date")
plt.ylabel("Index Value")

plt.legend()
plt.grid(True)

plt.tight_layout()

plt.savefig(
    CHART_DIR / "nifty50_historical_price.png",
    dpi=300
)

plt.show()

NameError: name 'nifty' is not defined

<Figure size 1600x600 with 0 Axes>

In [9]:
validation_df.to_csv(
    REPORT_DIR / "data_validation_report.csv",
    index=False
)

quality_df.to_csv(
    REPORT_DIR / "data_quality_report.csv",
    index=False
)

print("Validation reports saved successfully!")

OSError: Cannot save file into a non-existent directory: 'D:\BharatMarket_AI\outputs\reports'

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import scipy
import sklearn
import statsmodels
import xgboost

print("All libraries imported successfully!")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

All libraries imported successfully!
Pandas: 3.1.1
NumPy: 2.5.5
Scikit-learn: 1.9.1
